In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import pingouin as pg
import numpy as np
import sklearn as sk
from scipy.stats import boxcox
from scipy.stats import rankdata
from statsmodels.stats.contingency_tables import mcnemar
from itertools import combinations
from statsmodels.stats.multitest import multipletests


In [2]:
# Replace with our file path
df = pd.read_csv("ai_grading_final.csv")

df["prompt_type"] = pd.Categorical(df["prompt_type"], 
    categories=["very_pessimistic", "pessimistic", "neutral", "confident", "very_confident"], 
    ordered=True)

# Required columns:
required_columns = [
    "answer_key_id",
    "true_mistakes",
    "prompt_type",
    "ai_estimated_mistakes"
]
df = df.dropna()
# Dependent variable: AI grading error
df["ai_error"] = df["ai_estimated_mistakes"] - df["true_mistakes"]
df["Absolute_error"] = abs(df["ai_estimated_mistakes"]-df["true_mistakes"])

df["true_mistakes"] = df["true_mistakes"].astype("category")
df["prompt_type"] = df["prompt_type"].astype("category")

In [25]:
# 1. Aggreger til ét tal per sheet
df_agg = df.groupby(["answer_key_id", "true_mistakes"], as_index=False).agg(
    ai_error=("ai_error", "mean"),
    Absolute_error=("Absolute_error", "mean")
)

In [30]:
# ── Kruskal-Wallis for true_mistakes ───────────────────────────────────────────

# Med ai_error (signed)
kw_signed = pg.kruskal(
    data=df_agg,
    dv="ai_error",
    between="true_mistakes"
)
print("=== Kruskal-Wallis: ai_error (signed) ===")
print(kw_signed)

# Med Absolute_error
kw_absolute = pg.kruskal(
    data=df_agg,
    dv="Absolute_error",
    between="true_mistakes"
)
print("\n=== Kruskal-Wallis: Absolute_error ===")
print(kw_absolute)

=== Kruskal-Wallis: ai_error (signed) ===
                Source  ddof1         H         p_unc
Kruskal  true_mistakes      7  94.03576  1.830968e-17

=== Kruskal-Wallis: Absolute_error ===
                Source  ddof1          H     p_unc
Kruskal  true_mistakes      7  15.777048  0.027233


In [33]:
# Post-hoc Dunn's test (Bonferroni-korrigeret) for signed error
posthoc_signed_true = pg.pairwise_tests(
    data=df_agg,
    dv="ai_error",
    between="true_mistakes",
    parametric=False,  # bruger Mann-Whitney U til parvise tests
    padjust="holm"
)
print("\n=== Post-hoc (ai_error) ===")
display(posthoc_signed_true)
display(posthoc_signed_true[posthoc_signed_true["p_corr"] < 0.05][["A", "B", "p_corr"]])

# Post-hoc for Absolute_error
posthoc_absolute_true = pg.pairwise_tests(
    data=df_agg,
    dv="Absolute_error",
    between="true_mistakes",
    parametric=False,
    padjust="holm"
)
print("\n=== Post-hoc (Absolute_error) ===")
display(posthoc_absolute_true)
display(posthoc_absolute_true[posthoc_absolute_true["p_corr"] < 0.05][["A", "B", "p_corr"]])


=== Post-hoc (ai_error) ===


c:\Users\sebhj\miniforge3\envs\intelligent-systems\Lib\site-packages\pingouin\effsize.py:830: RuntimeWarning: Degrees of freedom <= 0 for slice
  poolsd = np.sqrt(((nx - 1) * x.var(ddof=1) + (ny - 1) * y.var(ddof=1)) / dof)
c:\Users\sebhj\miniforge3\envs\intelligent-systems\Lib\site-packages\numpy\_core\_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


,Contrast,A,B,Paired,Parametric,U_val,alternative,p_unc,p_corr,p_adjust,hedges
0,true_mistakes,0,2,False,False,19.0,two-sided,8.698324e-01,1.000000e+00,holm,NaN
1,true_mistakes,0,4,False,False,26.0,two-sided,9.422359e-01,1.000000e+00,holm,NaN
2,true_mistakes,0,6,False,False,32.0,two-sided,6.126250e-01,1.000000e+00,holm,NaN
3,true_mistakes,0,8,False,False,37.0,two-sided,4.225159e-01,1.000000e+00,holm,NaN
4,true_mistakes,0,10,False,False,39.0,two-sided,3.477513e-01,1.000000e+00,holm,NaN
5,true_mistakes,0,12,False,False,45.0,two-sided,1.400244e-01,1.000000e+00,holm,NaN
6,true_mistakes,0,14,False,False,15.0,two-sided,1.286037e-01,1.000000e+00,holm,NaN
7,true_mistakes,2,4,False,False,1219.0,two-sided,1.769227e-01,1.000000e+00,holm,0.302338
8,true_mistakes,2,6,False,False,1400.0,two-sided,4.679726e-03,6.551616e-02,holm,0.582383
9,true_mistakes,2,8,False,False,1589.5,two-sided,4.411427e-05,8.822855e-04,holm,0.784126


,A,B,p_corr
9,2,8,8.822855e-04
10,2,10,6.755527e-06
11,2,12,1.104284e-09
12,2,14,5.407050e-07
14,4,8,2.743390e-02
15,4,10,5.001917e-04
16,4,12,2.804994e-08
17,4,14,1.558162e-06
20,6,12,9.396508e-06
21,6,14,2.122392e-05



=== Post-hoc (Absolute_error) ===


c:\Users\sebhj\miniforge3\envs\intelligent-systems\Lib\site-packages\pingouin\effsize.py:830: RuntimeWarning: Degrees of freedom <= 0 for slice
  poolsd = np.sqrt(((nx - 1) * x.var(ddof=1) + (ny - 1) * y.var(ddof=1)) / dof)
c:\Users\sebhj\miniforge3\envs\intelligent-systems\Lib\site-packages\numpy\_core\_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


,Contrast,A,B,Paired,Parametric,U_val,alternative,p_unc,p_corr,p_adjust,hedges
0,true_mistakes,0,2,False,False,17.0,two-sided,0.744457,1.000000,holm,NaN
1,true_mistakes,0,4,False,False,19.0,two-sided,0.717998,1.000000,holm,NaN
2,true_mistakes,0,6,False,False,20.0,two-sided,0.770570,1.000000,holm,NaN
3,true_mistakes,0,8,False,False,12.0,two-sided,0.380251,1.000000,holm,NaN
4,true_mistakes,0,10,False,False,10.0,two-sided,0.302394,1.000000,holm,NaN
5,true_mistakes,0,12,False,False,13.0,two-sided,0.448129,1.000000,holm,NaN
6,true_mistakes,0,14,False,False,1.0,two-sided,0.192404,1.000000,holm,NaN
7,true_mistakes,2,4,False,False,1133.5,two-sided,0.517356,1.000000,holm,0.227953
8,true_mistakes,2,6,False,False,1176.0,two-sided,0.318211,1.000000,holm,0.345142
9,true_mistakes,2,8,False,False,1022.0,two-sided,0.676933,1.000000,holm,0.097328


,A,B,p_corr


Mann-Whitney U-testen i stedet for uafhængige t-test

In [34]:
alpha_corrected2 = 0.05 

results_sign_true = posthoc_signed_true.copy()
n1 = 50  # juster til jeres faktiske gruppestørrelser
n2 = 50

results_sign_true["z"] = (
    (results_sign_true["U_val"] - (n1*n2)/2)
    / np.sqrt(n1*n2*(n1+n2+1)/12)
)
results_sign_true["d"] = 2 * results_sign_true["z"].abs() / np.sqrt(n1+n2)

results_sign_true["required_n"] = results_sign_true["d"].apply(
    lambda d: np.ceil(
        pg.power_ttest(
            d=d,
            n=None,
            power=0.80,
            alpha=alpha_corrected2,
            contrast="two-samples",
            alternative="two-sided"
        ))).astype(int)

display(results_sign_true[["A", "B", "U_val", "d", "required_n"]])

print("Maximum required answer sheets (true_mistakes):", results_sign_true["required_n"].max())




alpha_corrected2 = 0.05

results_true_abs = posthoc_absolute_true.copy()

n1 = 50  # juster til jeres faktiske gruppestørrelser
n2 = 50

results_true_abs["z"] = (
    (results_true_abs["U_val"] - (n1*n2)/2)
    / np.sqrt(n1*n2*(n1+n2+1)/12)
)
results_true_abs["d"] = 2 * results_true_abs["z"].abs() / np.sqrt(n1+n2)


results_true_abs["required_n"] = results_true_abs["d"].apply(
    lambda d: np.ceil(
        pg.power_ttest(
            d=d,
            n=None,
            power=0.80,
            alpha=alpha_corrected2,
            contrast="two-samples",
            alternative="two-sided"
        ))).astype(int)

display(results_true_abs[["A", "B", "U_val", "d", "required_n"]])

print("Maximum required answer sheets (true_mistakes):", results_true_abs["required_n"].max())

,A,B,U_val,d,required_n
0,0,2,19.0,1.697258,7
1,0,4,26.0,1.687607,7
2,0,6,32.0,1.679335,7
3,0,8,37.0,1.672441,7
4,0,10,39.0,1.669683,7
5,0,12,45.0,1.661411,7
6,0,14,15.0,1.702774,7
7,2,4,1219.0,0.042742,8594
8,2,6,1400.0,0.206815,368
9,2,8,1589.5,0.468090,73


Maximum required answer sheets (true_mistakes): 8594


,A,B,U_val,d,required_n
0,0,2,17.0,1.700016,7
1,0,4,19.0,1.697258,7
2,0,6,20.0,1.695880,7
3,0,8,12.0,1.706910,7
4,0,10,10.0,1.709667,7
5,0,12,13.0,1.705531,7
6,0,14,1.0,1.722076,7
7,2,4,1133.5,0.160626,610
8,2,6,1176.0,0.102029,1509
9,2,8,1022.0,0.314358,160


Maximum required answer sheets (true_mistakes): 8257657
